# 0.1 Import Libraries

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tifffile

from bioimage_pipeline.data import (
    find_segmentation_gt_files,
    find_tiff_files,
    get_segmentation_gt_slice_index,
    load_gt_mask,
    load_tiff_image,
    match_prediction_to_gt_frame,
)
from bioimage_pipeline.evaluation import dice_coefficient, iou_score, precision_recall_f1
from bioimage_pipeline.preprocessing import denoise_image, normalize_intensity, subtract_background
from bioimage_pipeline.segmentation import clean_binary_mask, label_components, otsu_segmentation
from bioimage_pipeline.visualization import overlay_mask_on_image, save_side_by_side_segmentation

# 0.2 Set Paths

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATASET_DIR = PROJECT_ROOT / "data" / "raw" / "Fluo-C3DL-MDA231"
SEQUENCE_ID = "01"
OUTPUT_DIR = PROJECT_ROOT / "reports" / "milestone_2" / "notebook"

# 1.1 Load Sample Image

In [ ]:
frame_files = find_tiff_files(DATASET_DIR / SEQUENCE_ID, recursive=False)
if not frame_files:
    raise FileNotFoundError("No TIFF frames found in the selected sequence.")

sample_path = frame_files[0]
raw_image = load_tiff_image(sample_path)
sample_path, raw_image.shape, raw_image.dtype

# 1.2 Load Available Ground Truth Mask

In [ ]:
gt_files = find_segmentation_gt_files(DATASET_DIR, SEQUENCE_ID)
gt_matches = match_prediction_to_gt_frame([sample_path], gt_files)
matched_gt_paths = gt_matches.get(sample_path.resolve(), [])
gt_masks = [load_gt_mask(path) for path in matched_gt_paths]
[(path.name, mask.shape) for path, mask in zip(matched_gt_paths, gt_masks)]

# 2.1 Normalize Intensity

In [ ]:
normalized_image = normalize_intensity(raw_image, lower=1, upper=99)
normalized_image.shape, normalized_image.dtype, normalized_image.min(), normalized_image.max()

# 2.2 Denoise Image

In [ ]:
denoised_image = denoise_image(normalized_image, sigma=1.0)

# 2.3 Background Correction

In [ ]:
background_corrected = subtract_background(denoised_image, sigma=10.0)
processed_image = normalize_intensity(background_corrected, method="minmax")

# 3.1 Otsu Segmentation

In [ ]:
initial_mask = otsu_segmentation(processed_image)

# 3.2 Mask Cleanup

In [ ]:
binary_mask = clean_binary_mask(initial_mask, min_size=64)

# 3.3 Connected Component Labeling

In [ ]:
labeled_mask = label_components(binary_mask)
int(labeled_mask.max())

# 4.1 Visualize Segmentation Overlay

In [ ]:
figure, axis = plt.subplots(figsize=(7, 7))
overlay_mask_on_image(raw_image, binary_mask, ax=axis, title="Otsu Baseline")
plt.show()

# 5.1 Evaluate Against Ground Truth

In [ ]:
true_parts = []
predicted_parts = []
for gt_path, gt_mask in zip(matched_gt_paths, gt_masks):
    if gt_mask.shape == binary_mask.shape:
        prediction_for_gt = binary_mask
    elif gt_mask.ndim == 2 and binary_mask.ndim == 3:
        z_index = get_segmentation_gt_slice_index(gt_path)
        if z_index is None or not 0 <= z_index < binary_mask.shape[0]:
            print(f"Skipping GT with invalid z-index: {gt_path.name}")
            continue
        prediction_for_gt = binary_mask[z_index]
    else:
        print(f"Skipping incompatible GT shape: {gt_path.name} {gt_mask.shape}")
        continue
    true_parts.append((gt_mask != 0).ravel())
    predicted_parts.append((prediction_for_gt != 0).ravel())

if not true_parts:
    print("No compatible segmentation GT is available; evaluation skipped.")
else:
    y_true = np.concatenate(true_parts)
    y_pred = np.concatenate(predicted_parts)
    metrics = {
        "evaluation_scope": "annotated_2d_planes" if gt_masks[0].ndim == 2 else "full_volume",
        "evaluated_gt_files": len(true_parts),
        "dice": dice_coefficient(y_true, y_pred),
        "iou": iou_score(y_true, y_pred),
        **precision_recall_f1(y_true, y_pred),
    }
    print(metrics)

# 6.1 Save Outputs

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
overlay_path = save_side_by_side_segmentation(
    raw_image,
    processed_image,
    binary_mask,
    OUTPUT_DIR / "sample_otsu_overlay.png",
)
mask_path = OUTPUT_DIR / "sample_otsu_labels.tif"
tifffile.imwrite(mask_path, labeled_mask.astype("uint16"), photometric="minisblack")
overlay_path, mask_path

# 7.1 Summary and Next Steps

Review preprocessing behavior, segmentation errors, and GT coverage before tuning parameters or introducing learned models.